In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# =========================
# Style & Palette (unchanged)
# =========================
STYLE = {
    "w": 1200, "h": 700,
    "line_width": 4, "event_line": 2, "axis_line": 2,
    "grid": 0.35,
    "tick_font": 16, "axis_title": 26, "panel_title": 30,
    "legend_font": 20, "marker_size": 12, "caption": 16, "main_title": 32
}
PALETTE = {
    "process": "#206A92", "artifact": "#1E5631", "inputOutput": "#A9A9A9",
    "startEnd": "#E3C800", "stroke": "#2E2E2E", "bg": "#FFFFFF",
}


# =========================
# Data generation helpers
# =========================
def generate_arps(t: np.ndarray, qi: float = 1000.0, Di: float = 0.5, b: float = 0.5) -> np.ndarray:
    """Return hyperbolic (Arps) decline series q(t)."""
    return qi * (1 + b * Di * t) ** (-1.0 / b)


def generate_messy_series(
    t: np.ndarray,
    qi: float = 1000.0,
    Di: float = 0.5,
    b: float = 0.5,
    noise_level: float = 0.03,
    events: list[tuple[float, float]] | None = None,
    outage_centers: list[float] | None = None,
    rng_seed: int = 42,
):
    """
    Build 'messy' production: Arps base + noise + step events + brief outages.
    Returns (series, event_x, event_y, event_text).
    """
    if events is None:
        events = [(2.0, -120), (3.5, 90), (4.2, -200), (6.0, 70), (7.1, -150), (8.5, 110)]
    if outage_centers is None:
        outage_centers = [5.0, 7.8]

    rng = np.random.default_rng(rng_seed)
    base = generate_arps(t, qi, Di, b)
    noise = rng.normal(0, qi * noise_level, size=t.shape)
    y = base + noise

    event_x, event_y, event_text = [], [], []
    for te, amp in events:
        idx = int(np.abs(t - te).argmin())
        y[idx:] += amp
        event_x.append(t[idx])
        event_y.append(y[idx])
        event_text.append(f"Step {'down' if amp < 0 else 'up'} ({amp:+.0f})")

    for oc in outage_centers:
        mask = (t > oc - 0.05) & (t < oc + 0.05)
        y[mask] = np.maximum(y[mask] * 0.1, 1.0)

    return y, event_x, event_y, event_text


# =========================
# Figure building helpers
# =========================
def add_arps_panel(fig, t, q):
    """Add Arps line + subtle underfill to the left panel."""
    fig.add_trace(
        go.Scatter(
            x=t, y=q, mode="lines",
            name="Arps decline (hyperbolic)",
            line=dict(color=PALETTE["process"], width=STYLE["line_width"], shape="spline"),
            hovertemplate="t=%{x:.2f} yr<br>q=%{y:.1f}<extra>Arps</extra>",
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=np.concatenate([t, t[::-1]]),
            y=np.concatenate([q, np.zeros_like(q)]),
            mode="lines",
            fill="toself",
            hoverinfo="skip",
            line=dict(color="rgba(0,0,0,0)"),
            fillcolor="rgba(32,106,146,0.08)",
            showlegend=False,
        ),
        row=1, col=1,
    )


def add_messy_panel(fig, t, y, event_x, event_y, event_text, outage_centers):
    """Add messy production, event lines/markers, and outage bands to the right panel."""
    fig.add_trace(
        go.Scatter(
            x=t, y=y, mode="lines",
            name="Observed production",
            line=dict(color=PALETTE["artifact"], width=STYLE["line_width"], shape="spline"),
            hovertemplate="t=%{x:.2f} yr<br>q=%{y:.1f}<extra>Observed</extra>",
        ),
        row=1, col=2,
    )

    # vertical event lines
    for xv in event_x:
        fig.add_vline(
            x=xv,
            line=dict(color=PALETTE["startEnd"], width=STYLE["event_line"], dash="dash"),
            row=1, col=2,
        )

    # event markers
    fig.add_trace(
        go.Scatter(
            x=event_x, y=event_y, mode="markers",
            name="Operational events",
            marker=dict(size=STYLE["marker_size"], color=PALETTE["startEnd"], line=dict(width=0)),
            hovertext=event_text, hoverinfo="text",
        ),
        row=1, col=2,
    )

    # outage bands
    for oc in outage_centers:
        fig.add_vrect(
            x0=oc - 0.05, x1=oc + 0.05,
            fillcolor=PALETTE["inputOutput"], opacity=0.15, line_width=0, row=1, col=2
        )


def style_axes(fig, ymax: float):
    """Apply consistent axes styling to both panels."""
    for c in (1, 2):
        fig.update_xaxes(
            title=dict(text="Time (years)", font=dict(size=STYLE["axis_title"])),
            range=[0, 10],
            showgrid=True, gridcolor=PALETTE["inputOutput"], gridwidth=1,
            showline=True, linewidth=STYLE["axis_line"], linecolor=PALETTE["stroke"],
            ticks="outside", ticklen=12, tickwidth=1, tickcolor=PALETTE["stroke"],
            tickfont=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]),
            row=1, col=c,
        )
        fig.update_yaxes(
            title=dict(text="Rate BBL/Day", font=dict(size=STYLE["axis_title"])),
            range=[0, ymax],
            showgrid=True, gridcolor=PALETTE["inputOutput"], gridwidth=1,
            showline=True, linewidth=STYLE["axis_line"], linecolor=PALETTE["stroke"],
            ticks="outside", ticklen=6, tickwidth=1, tickcolor=PALETTE["stroke"],
            tickfont=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]),
            row=1, col=c,
        )


def add_challenge_captions(fig):
    """Add two concise challenge captions below the panels."""
    captions = [
        {
            "x": 0, "xanchor": "left", "align": "left",
            "text": "<b>Challenge:</b> Idealized declines often ignore operational disruptions<br>"
                    "and changing reservoir conditions."
        },
        {
            "x": 1, "xanchor": "right", "align": "right",
            "text": "<b>Challenge:</b> Real-world events distort the signal, making naïve<br>"
                    "extrapolation unreliable for long-term forecasts."
        },
    ]
    for cap in captions:
        fig.add_annotation(
            xref="paper", yref="paper",
            y=-0.3, yanchor="top",
            showarrow=False,
            font=dict(size=STYLE["caption"], color=PALETTE["stroke"]),
            **cap,
        )


def set_panel_title_fonts(fig, panel_titles):
    """Ensure subplot titles use the configured panel title size/color."""
    for ann in fig.layout.annotations:
        if getattr(ann, "text", None) in panel_titles:
            ann.font.size = STYLE["panel_title"]
            ann.font.color = PALETTE["stroke"]


# =========================
# Build figure
# =========================
# Time grids
t = np.linspace(0, 10, 400)
t2 = np.linspace(0, 10, 400)

# Series
q = generate_arps(t)
messy, event_x, event_y, event_text = generate_messy_series(
    t2, events=[(2.0, -120), (3.5, 90), (4.2, -200), (6.0, 70), (7.1, -150), (8.5, 110)],
    outage_centers=[5.0, 7.8]
)

ymax = float(max(np.max(q), np.max(messy)) * 1.1)

# Subplot titles
panel_titles = [
    "<b>Classic DCA (Arps)</b><br>Idealized Decline",
    "<b>Real Production Data</b><br>Drops, Spikes, Outages, and Noise",
]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=panel_titles)

# Traces
add_arps_panel(fig, t, q)
add_messy_panel(fig, t2, messy, event_x, event_y, event_text, outage_centers=[5.0, 7.8])

# Axes & captions
style_axes(fig, ymax)
add_challenge_captions(fig)

# Layout (main title slightly higher: y=0.98)
fig.update_layout(
    title=dict(
        text="<b>Idealized vs. Real-World Production Data</b>",
        y=0.98, x=0.5, xanchor="center", yanchor="top",
        font=dict(size=STYLE["main_title"], color=PALETTE["stroke"]),
    ),
    width=STYLE["w"], height=STYLE["h"],
    paper_bgcolor=PALETTE["bg"], plot_bgcolor=PALETTE["bg"],
    margin=dict(l=80, r=50, t=110, b=210),
    legend=dict(
        orientation="h", yanchor="bottom", y=-0.29, xanchor="center", x=0.5,
        font=dict(size=STYLE["legend_font"], color=PALETTE["stroke"]),
        bgcolor="rgba(255,255,255,0)",
    ),
    showlegend=True,
)

# Ensure subplot title styling
set_panel_title_fonts(fig, panel_titles)

fig.show()


In [ ]:
"""
Volve F-12 | Exponential Decay (Closed-Loop) + Scenario Curves (Plotly)

- Fits P_reservoir and decay_rate on the first 40% of the data.
- Forecasts the remainder in CLOSED-LOOP (no future PI/Pwf).
- Overlays 4 deterministic scenario curves with varied parameters.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------- Style & palette ----------
STYLE = {
    "w": 1200, "h": 700,
    "line_width": 4, "event_line": 2, "axis_line": 2,
    "grid": 0.35,
    "tick_font": 16, "axis_title": 26, "panel_title": 30,
    "legend_font": 20, "marker_size": 12, "caption": 16, "main_title": 32
}
PALETTE = {
    "process": "#206A92", "artifact": "#1E5631", "inputOutput": "#A9A9A9",
    "startEnd": "#E3C800", "stroke": "#2E2E2E", "bg": "#FFFFFF",
}

# ---------- Model ----------
class ExponentialDecayStrategy:
    def __init__(self, P_reservoir: float, decay_rate: float):
        self.P_reservoir = float(P_reservoir)
        self.decay_rate = float(decay_rate)

    def compute_Q_phys(self, PI_measured: np.ndarray, P_wf: np.ndarray, t_feature: np.ndarray) -> np.ndarray:
        P_res_t = self.P_reservoir * np.exp(-self.decay_rate * t_feature)
        return np.abs(PI_measured * (P_res_t - P_wf))

# ---------- Data helpers ----------
def extract_series(df: pd.DataFrame):
    t = df["Tempo_Inicio_Prod"].to_numpy(dtype=float)
    q_actual = df["BORE_OIL_VOL"].to_numpy(dtype=float)
    PI_measured = df["PI"].to_numpy(dtype=float)
    P_wf = df["AVG_DOWNHOLE_PRESSURE"].to_numpy(dtype=float)
    return t, q_actual, PI_measured, P_wf

def train_test_split_index(n: int, train_frac: float = 0.40) -> int:
    return max(1, int(np.floor(n * train_frac)))

# ---------- Fitting ----------
def q_model_wrapper(xdata: tuple[np.ndarray, np.ndarray, np.ndarray], P_reservoir: float, decay_rate: float) -> np.ndarray:
    t, PI, P_wf = xdata
    model = ExponentialDecayStrategy(P_reservoir=P_reservoir, decay_rate=decay_rate)
    return model.compute_Q_phys(PI_measured=PI, P_wf=P_wf, t_feature=t)

def initial_guess(t: np.ndarray, q: np.ndarray, PI: np.ndarray, P_wf: np.ndarray) -> tuple[float, float]:
    m = max(10, min(50, int(0.05 * t.size)))
    idx = slice(0, m)
    PI_safe = np.where(PI[idx] <= 1e-6, 1e-6, PI[idx])
    P_res0 = float(np.median(q[idx] / PI_safe + P_wf[idx]))
    decay0 = 5e-3
    P_res0 = float(np.clip(P_res0, 0.1 * np.nanmedian(P_wf), 10.0 * np.nanmax(P_wf)))
    return P_res0, decay0

def fit_parameters(t: np.ndarray, q: np.ndarray, PI: np.ndarray, P_wf: np.ndarray) -> tuple[float, float, np.ndarray]:
    p0 = initial_guess(t, q, PI, P_wf)
    bounds = ([1e-3, 1e-8], [1e6, 1.0])
    popt, pcov = curve_fit(
        q_model_wrapper, xdata=(t, PI, P_wf), ydata=q,
        p0=p0, bounds=bounds, maxfev=20000
    )
    return float(popt[0]), float(popt[1]), pcov

# ---------- Closed-loop forecast ----------
def predict_closed_loop(
    P_res: float,
    decay: float,
    t: np.ndarray,
    PI: np.ndarray,
    P_wf: np.ndarray,
    split_idx: int,
    *,
    n_ahead: int = 0,
    pi_mode: str = "hold_last",   # "hold_last" or "rolling_mean"
    pwf_mode: str = "hold_last",  # "hold_last" or "rolling_mean"
    window: int = 30,
) -> tuple[np.ndarray, np.ndarray]:
    t0 = float(t[split_idx - 1])
    dt = float(np.median(np.diff(t))) if len(t) > 1 else 1.0
    if n_ahead > 0:
        t_future_extra = t[-1] + dt * np.arange(1, n_ahead + 1)
        t_full = np.concatenate([t, t_future_extra])
    else:
        t_full = t.copy()

    def _scenario(arr):
        if (pi_mode == "rolling_mean") or (pwf_mode == "rolling_mean"):
            w = max(1, min(window, split_idx))
            return np.full_like(t_full[split_idx:], np.nanmean(arr[split_idx - w: split_idx]))
        else:
            return np.full_like(t_full[split_idx:], arr[split_idx - 1])

    PI_tail = _scenario(PI)
    Pwf_tail = _scenario(P_wf)

    P_res_t0 = P_res * np.exp(-decay * t0)
    dt_tail = t_full[split_idx:] - t0
    P_res_future = P_res_t0 * np.exp(-decay * dt_tail)
    q_tail = np.abs(PI_tail * (P_res_future - Pwf_tail))

    q_pred = np.full_like(t_full, np.nan, dtype=float)  # NaN before split for clean plotting
    q_pred[split_idx:] = q_tail
    return t_full, q_pred

# ---------- Plotting ----------
def add_observed_data_panel(fig, t, q_actual):
    fig.add_trace(
        go.Scatter(
            x=t, y=q_actual, mode="lines",
            name="Observed Production",
            line=dict(color=PALETTE["artifact"], width=STYLE["line_width"], shape="spline"),
            hovertemplate="t=%{x:.0f} d<br>q=%{y:.1f}<extra>Observed</extra>",
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([t, t[::-1]]),
            y=np.concatenate([q_actual, np.zeros_like(q_actual)]),
            mode="lines", fill="toself", hoverinfo="skip",
            line=dict(color="rgba(0,0,0,0)"),
            fillcolor="rgba(30,86,49,0.08)", showlegend=False,
        ),
        row=1, col=1
    )

def style_axes(fig, t_range: tuple[float, float], y_max: float):
    for c in (1, 2):
        fig.update_xaxes(
            title=dict(text="Time (days)", font=dict(size=STYLE["axis_title"])),
            range=list(t_range), showgrid=True, gridcolor=PALETTE["inputOutput"], gridwidth=1,
            showline=True, linewidth=STYLE["axis_line"], linecolor=PALETTE["stroke"],
            ticks="outside", ticklen=12, tickwidth=1, tickcolor=PALETTE["stroke"],
            tickfont=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]),
            zeroline=False, row=1, col=c,
        )
        fig.update_yaxes(
            title=dict(text="Oil Rate (BBL/Day)", font=dict(size=STYLE["axis_title"])),
            range=[0, y_max], showgrid=True, gridcolor=PALETTE["inputOutput"], gridwidth=1,
            showline=True, linewidth=STYLE["axis_line"], linecolor=PALETTE["stroke"],
            ticks="outside", ticklen=6, tickwidth=1, tickcolor=PALETTE["stroke"],
            tickfont=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]),
            zeroline=False, row=1, col=c,
        )

def set_panel_title_fonts(fig, panel_titles: list[str]):
    for ann in fig.layout.annotations:
        if getattr(ann, "text", None) in panel_titles:
            ann.font.size = STYLE["panel_title"]
            ann.font.color = PALETTE["stroke"]

def build_figure_closed_loop(
    t: np.ndarray,
    q_actual: np.ndarray,
    q_pred_tail: np.ndarray,
    split_idx: int,
    scenario_curves: list[tuple[str, np.ndarray, str]] | None = None,  # (label, q_pred, dash)
) -> go.Figure:
    panel_titles = ["<b>Observed Production Data</b>", "<b>Closed-Loop Forecast vs. Observed</b>"]
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=panel_titles)

    # Left = observed
    add_observed_data_panel(fig, t, q_actual)

    # Right = observed + base forecast
    fig.add_trace(
        go.Scatter(
            x=t, y=q_actual, mode="lines",
            name="Observed Production",
            line=dict(color=PALETTE["artifact"], width=STYLE["line_width"], shape="spline"),
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(
            x=t, y=q_pred_tail, mode="lines",
            name="Closed-Loop Forecast",
            line=dict(color=PALETTE["process"], width=STYLE["line_width"], shape="spline"),
        ),
        row=1, col=2
    )

    # Optional extra scenario curves
    if scenario_curves:
        for label, q_curve, dash in scenario_curves:
            fig.add_trace(
                go.Scatter(
                    x=t, y=q_curve, mode="lines", name=label,
                    line=dict(color=PALETTE["process"], width=2, dash=dash, shape="spline"),
                ),
                row=1, col=2
            )

    # Training split marker & training shading (both panels)
    t_split = float(t[split_idx - 1]) if split_idx > 0 else float(t[0])
    for c in (1, 2):
        fig.add_vline(x=t_split, line=dict(color=PALETTE["startEnd"], width=STYLE["event_line"], dash="dash"), row=1, col=c)
        fig.add_vrect(x0=t.min(), x1=t_split, fillcolor=PALETTE["inputOutput"], opacity=0.12, line_width=0, row=1, col=c)

    # Axes + layout
    y_max = float(np.nanmax([q_actual, q_pred_tail] + ([c[1] for c in scenario_curves] if scenario_curves else [])) * 1.10)
    style_axes(fig, (float(t.min()), float(t.max())), y_max)

    # Region labels (both panels): Training vs Extrapolation
    x_train_mid = t.min() + 0.5 * (t_split - t.min())
    x_extrap_mid = t_split + 0.5 * (t.max() - t_split)
    for c in (1, 2):
        fig.add_annotation(x=x_train_mid, y=y_max * 0.97, text="<b>Training</b>",
                           showarrow=False, font=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]),
                           row=1, col=c)
        fig.add_annotation(x=x_extrap_mid, y=y_max * 0.97, text="<b>Extrapolation</b>",
                           showarrow=False, font=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]),
                           row=1, col=c)

    fig.update_layout(
        title=dict(
            text="<b>Volve F-12: Exponential Pressure Decay — Closed-Loop Forecast</b>",
            y=0.98, x=0.5, xanchor="center", yanchor="top",
            font=dict(size=STYLE["main_title"], color=PALETTE["stroke"]),
        ),
        width=STYLE["w"], height=STYLE["h"],
        paper_bgcolor=PALETTE["bg"], plot_bgcolor=PALETTE["bg"],
        margin=dict(l=80, r=50, t=110, b=60),
        legend=dict(
            orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5,
            font=dict(size=STYLE["legend_font"], color=PALETTE["stroke"]),
            bgcolor="rgba(255,255,255,0)",
        ),
        showlegend=True,
    )
    set_panel_title_fonts(fig, panel_titles)
    return fig

# =========================
# New strategies (NumPy)
# =========================
class ArpsDeclineStrategy:
    """Hyperbolic decline: P_res(t) = P0 / (1 + b*D*t)^(1/b)."""
    def __init__(self, P_reservoir: float, decay_rate: float, b_factor: float):
        self.P_reservoir = float(P_reservoir)
        self.decay_rate = float(decay_rate)
        self.b_factor = float(b_factor)

    def compute_Q_phys(self, PI_measured: np.ndarray, P_wf: np.ndarray, t_feature: np.ndarray) -> np.ndarray:
        denom = (1.0 + self.b_factor * self.decay_rate * t_feature)
        P_res_t = self.P_reservoir / np.power(denom, 1.0 / self.b_factor)
        return np.abs(PI_measured * (P_res_t - P_wf))


class StaticPressureStrategy:
    """Constant reservoir pressure scenario (no depletion)."""
    def __init__(self, P_reservoir: float):
        self.P_reservoir = float(P_reservoir)

    def compute_Q_phys(self, PI_measured: np.ndarray, P_wf: np.ndarray, t_feature: np.ndarray) -> np.ndarray:
        q_base = np.abs(PI_measured * (self.P_reservoir - P_wf))
        # broadcast to t_feature length (if needed)
        if q_base.ndim == 1 and t_feature.ndim == 1 and q_base.shape[0] != t_feature.shape[0]:
            q_base = q_base * np.ones_like(t_feature, dtype=float)
        return q_base


class DynamicEnsembleStrategy:
    """
    Average of exponential & static pressure responses:
    Q = ( |PI*(P0*exp(-D*t)-Pwf)| + |PI*(P0 - Pwf)| ) / 2
    """
    def __init__(self, P_reservoir: float, decay_rate: float, absolute_value: bool = True):
        self.P_reservoir = float(P_reservoir)
        self.decay_rate = float(decay_rate)
        self.absolute_value = bool(absolute_value)

    def compute_Q_phys(self, PI_measured: np.ndarray, P_wf: np.ndarray, t_feature: np.ndarray) -> np.ndarray:
        P_res_t = self.P_reservoir * np.exp(-self.decay_rate * t_feature)
        if self.absolute_value:
            q1 = np.abs(PI_measured * (P_res_t - P_wf))
            q2 = np.abs(PI_measured * (self.P_reservoir - P_wf))
        else:
            q1 = PI_measured * (P_res_t - P_wf)
            q2 = PI_measured * (self.P_reservoir - P_wf)
        return 0.5 * (q1 + q2)


class CombinedExpArpsStrategy:
    """Average of Exponential and Arps reservoir pressure models."""
    def __init__(self, P_reservoir: float, decay_rate: float, b_factor: float):
        self.exp = ExponentialDecayStrategy(P_reservoir, decay_rate)
        self.arps = ArpsDeclineStrategy(P_reservoir, decay_rate, b_factor)

    def compute_Q_phys(self, PI_measured: np.ndarray, P_wf: np.ndarray, t_feature: np.ndarray) -> np.ndarray:
        q_exp = self.exp.compute_Q_phys(PI_measured, P_wf, t_feature)
        q_arps = self.arps.compute_Q_phys(PI_measured, P_wf, t_feature)
        return 0.5 * (q_exp + q_arps)


class WeightedEnsembleStrategy:
    """Weighted blend of DynamicEnsemble and Exponential with fixed alpha in [0,1]."""
    def __init__(self, P_reservoir: float, decay_rate: float, alpha: float = 0.5):
        self.dyn = DynamicEnsembleStrategy(P_reservoir, decay_rate, absolute_value=True)
        self.exp = ExponentialDecayStrategy(P_reservoir, decay_rate)
        self.alpha = float(np.clip(alpha, 0.0, 1.0))

    def compute_Q_phys(self, PI_measured: np.ndarray, P_wf: np.ndarray, t_feature: np.ndarray) -> np.ndarray:
        q1 = self.dyn.compute_Q_phys(PI_measured, P_wf, t_feature)
        q2 = self.exp.compute_Q_phys(PI_measured, P_wf, t_feature)
        return self.alpha * q1 + (1.0 - self.alpha) * q2


# =========================
# Curve-fit wrappers & inits
# =========================
def q_model_wrapper_arps(xdata, P_reservoir, decay_rate, b_factor):
    t, PI, Pwf = xdata
    return ArpsDeclineStrategy(P_reservoir, decay_rate, b_factor).compute_Q_phys(PI, Pwf, t)

def q_model_wrapper_static(xdata, P_reservoir):
    t, PI, Pwf = xdata
    return StaticPressureStrategy(P_reservoir).compute_Q_phys(PI, Pwf, t)

def q_model_wrapper_dynamic(xdata, P_reservoir, decay_rate):
    t, PI, Pwf = xdata
    return DynamicEnsembleStrategy(P_reservoir, decay_rate).compute_Q_phys(PI, Pwf, t)

def q_model_wrapper_combined(xdata, P_reservoir, decay_rate, b_factor):
    t, PI, Pwf = xdata
    return CombinedExpArpsStrategy(P_reservoir, decay_rate, b_factor).compute_Q_phys(PI, Pwf, t)

def q_model_wrapper_weighted(xdata, P_reservoir, decay_rate, alpha):
    t, PI, Pwf = xdata
    return WeightedEnsembleStrategy(P_reservoir, decay_rate, alpha).compute_Q_phys(PI, Pwf, t)


def initial_guess_static(q, PI, Pwf):
    PI_safe = np.where(PI <= 1e-6, 1e-6, PI)
    P_res0 = float(np.median(q / PI_safe + Pwf))
    return max(P_res0, 1.0)

def initial_guess_arps(t, q, PI, Pwf):
    # Start from exponential guesses and a moderate b
    P_res0, decay0 = initial_guess(t, q, PI, Pwf)
    b0 = 0.5
    return P_res0, max(decay0, 1e-5), b0

def initial_guess_combined(t, q, PI, Pwf):
    P_res0, decay0 = initial_guess(t, q, PI, Pwf)
    b0 = 0.5
    return P_res0, decay0, b0

# =========================
# Build 5 deterministic scenario curves (per strategy)
# =========================
def make_strategy(name: str, params: dict):
    if name == "Exponential":
        return ExponentialDecayStrategy(params["P_reservoir"], params["decay_rate"])
    if name == "Arps":
        return ArpsDeclineStrategy(params["P_reservoir"], params["decay_rate"], params["b_factor"])
    if name == "Static":
        return StaticPressureStrategy(params["P_reservoir"])
    if name == "DynamicEnsemble":
        return DynamicEnsembleStrategy(params["P_reservoir"], params["decay_rate"])
    if name == "CombinedExpArps":
        return CombinedExpArpsStrategy(params["P_reservoir"], params["decay_rate"], params["b_factor"])
    if name == "WeightedEnsemble":
        return WeightedEnsembleStrategy(params["P_reservoir"], params["decay_rate"], params["alpha"])
    raise ValueError(f"Unknown strategy: {name}")


def build_scenario_curves(
    name: str, base_params: dict,
    t: np.ndarray, PI: np.ndarray, Pwf: np.ndarray, split_idx: int,
    n_ahead: int = 0
) -> list[tuple[str, np.ndarray, str]]:
    """
    Returns exactly 5 curves (label, q_curve, dash) around the base params.
    Prioritizes decay and P_reservoir variations; adds model-specific fifth.
    """
    p = base_params.copy()
    scenarios = []

    def predict_with_params(label, dash, **overrides):
        p_ = p.copy(); p_.update(overrides)
        strat = make_strategy(name, p_)
        _, q_curve = predict_closed_loop_with_strategy(
            strat, t, PI, Pwf, split_idx, n_ahead=n_ahead, pi_mode="hold_last", pwf_mode="hold_last"
        )
        scenarios.append((label, q_curve, dash))

    # Common tweaks
    if "decay_rate" in p:
        predict_with_params("Decay -20%", "dot", decay_rate=p["decay_rate"] * 0.80)
        predict_with_params("Decay +20%", "dash", decay_rate=p["decay_rate"] * 1.20)
    if "P_reservoir" in p:
        predict_with_params("P_res -10%", "dashdot", P_reservoir=p["P_reservoir"] * 0.90)
        predict_with_params("P_res +10%", "longdash", P_reservoir=p["P_reservoir"] * 1.10)

    # Fifth tweak depends on model
    if name in {"Arps", "CombinedExpArps"} and "b_factor" in p:
        predict_with_params("b +20%", "longdashdot", b_factor=p["b_factor"] * 1.20)
    elif name == "Static":
        predict_with_params("P_res +20%", "longdashdot", P_reservoir=p["P_reservoir"] * 1.20)
    elif name == "DynamicEnsemble":
        # both varied slightly
        predict_with_params("Both (+10% P_res, -10% D)", "longdashdot",
                            P_reservoir=p["P_reservoir"] * 1.10,
                            decay_rate=p["decay_rate"] * 0.90)
    elif name == "WeightedEnsemble":
        alpha_new = float(np.clip(p.get("alpha", 0.5) + 0.2, 0.0, 1.0))
        predict_with_params(f"alpha={alpha_new:.1f}", "longdashdot", alpha=alpha_new)

    # Ensure exactly 5; if fewer, duplicate nearest variations
    if len(scenarios) < 5:
        scenarios = (scenarios * ((5 + len(scenarios) - 1)//len(scenarios)))[:5]
    elif len(scenarios) > 5:
        scenarios = scenarios[:5]
    return scenarios


# =========================
# Generic closed-loop forecast
# =========================
def predict_closed_loop_with_strategy(
    strategy_obj,
    t: np.ndarray, PI: np.ndarray, Pwf: np.ndarray, split_idx: int,
    *, n_ahead: int = 0, pi_mode: str = "hold_last", pwf_mode: str = "hold_last", window: int = 30
) -> tuple[np.ndarray, np.ndarray]:
    t0 = float(t[split_idx - 1])
    dt = float(np.median(np.diff(t))) if len(t) > 1 else 1.0
    if n_ahead > 0:
        t_future_extra = t[-1] + dt * np.arange(1, n_ahead + 1)
        t_full = np.concatenate([t, t_future_extra])
    else:
        t_full = t.copy()

    def _scenario(arr):
        if (pi_mode == "rolling_mean") or (pwf_mode == "rolling_mean"):
            w = max(1, min(window, split_idx))
            return np.full_like(t_full[split_idx:], np.nanmean(arr[split_idx - w: split_idx]))
        else:
            return np.full_like(t_full[split_idx:], arr[split_idx - 1])

    PI_tail = _scenario(PI)
    Pwf_tail = _scenario(Pwf)

    # Compute tail using absolute time stamps on the tail portion
    q_tail = strategy_obj.compute_Q_phys(PI_tail, Pwf_tail, t_full[split_idx:])
    q_pred = np.full_like(t_full, np.nan, dtype=float)
    q_pred[split_idx:] = q_tail
    return t_full, q_pred

# =========================
# Fit helpers per strategy
# =========================
def fit_exponential(t_tr, q_tr, PI_tr, Pwf_tr) -> dict:
    P_res_opt, decay_opt, _ = fit_parameters(t_tr, q_tr, PI_tr, Pwf_tr)
    return {"P_reservoir": P_res_opt, "decay_rate": decay_opt}

def fit_static(t_tr, q_tr, PI_tr, Pwf_tr) -> dict:
    p0 = initial_guess_static(q_tr, PI_tr, Pwf_tr)
    popt, _ = curve_fit(
        q_model_wrapper_static, xdata=(t_tr, PI_tr, Pwf_tr), ydata=q_tr,
        p0=[p0], bounds=([1e-3], [1e6]), maxfev=20000
    )
    return {"P_reservoir": float(popt[0])}

def fit_arps(t_tr, q_tr, PI_tr, Pwf_tr) -> dict:
    P_res0, decay0, b0 = initial_guess_arps(t_tr, q_tr, PI_tr, Pwf_tr)
    bounds_lo = [1e-3, 1e-8, 1e-4]
    bounds_hi = [1e6, 1.0, 3.0]
    popt, _ = curve_fit(
        q_model_wrapper_arps, xdata=(t_tr, PI_tr, Pwf_tr), ydata=q_tr,
        p0=[P_res0, decay0, b0], bounds=(bounds_lo, bounds_hi), maxfev=30000
    )
    return {"P_reservoir": float(popt[0]), "decay_rate": float(popt[1]), "b_factor": float(popt[2])}

def fit_dynamic(t_tr, q_tr, PI_tr, Pwf_tr) -> dict:
    P_res0, decay0 = initial_guess(t_tr, q_tr, PI_tr, Pwf_tr)
    popt, _ = curve_fit(
        q_model_wrapper_dynamic, xdata=(t_tr, PI_tr, Pwf_tr), ydata=q_tr,
        p0=[P_res0, decay0], bounds=([1e-3, 1e-8], [1e6, 1.0]), maxfev=20000
    )
    return {"P_reservoir": float(popt[0]), "decay_rate": float(popt[1])}

def fit_combined(t_tr, q_tr, PI_tr, Pwf_tr) -> dict:
    P_res0, decay0, b0 = initial_guess_combined(t_tr, q_tr, PI_tr, Pwf_tr)
    popt, _ = curve_fit(
        q_model_wrapper_combined, xdata=(t_tr, PI_tr, Pwf_tr), ydata=q_tr,
        p0=[P_res0, decay0, b0], bounds=([1e-3, 1e-8, 1e-4], [1e6, 1.0, 3.0]), maxfev=30000
    )
    return {"P_reservoir": float(popt[0]), "decay_rate": float(popt[1]), "b_factor": float(popt[2])}

def fit_weighted(t_tr, q_tr, PI_tr, Pwf_tr) -> dict:
    P_res0, decay0 = initial_guess(t_tr, q_tr, PI_tr, Pwf_tr)
    alpha0 = 0.5
    popt, _ = curve_fit(
        q_model_wrapper_weighted, xdata=(t_tr, PI_tr, Pwf_tr), ydata=q_tr,
        p0=[P_res0, decay0, alpha0],
        bounds=([1e-3, 1e-8, 0.0], [1e6, 1.0, 1.0]), maxfev=30000
    )
    return {"P_reservoir": float(popt[0]), "decay_rate": float(popt[1]), "alpha": float(popt[2])}


# ---------- Main ----------
# =========================
# UPDATED main: fit & plot all models (each with 5 scenario curves)
# =========================
def main(
    df_f12: pd.DataFrame | None = None,
    train_frac: float = 0.40,
    load_via_pipeline: bool = True,
    dataset_name: str = "VOLVE",
    target_well: str = "15/9-F-14",
    n_ahead: int = 0,  # extend beyond dataset if desired
):
    # 1) Load data (same logic you already have)
    if df_f12 is not None:
        df = df_f12.copy()
        print("✅ Usando DataFrame fornecido (df_f12).")
    else:
        if not load_via_pipeline:
            raise ValueError("Forneça df_f12 ou habilite load_via_pipeline=True.")
        from common.config_wells import get_data_sources
        from data.data_loading import DataSource
        from common.batch_preprocessing import load_and_preprocess_data

        volve_config = next((ds for ds in get_data_sources() if ds.get("name") == dataset_name), None)
        if volve_config is None:
            raise ValueError(f"Config '{dataset_name}' não encontrada.")
        df = load_and_preprocess_data(DataSource, config=volve_config,
                                      selected_features=volve_config.get("features"),
                                      well=target_well)
        print(f"✅ Dados carregados: {len(df):,d} linhas.")

    # 2) Extract + split
    t, q, PI, Pwf = extract_series(df)
    split_idx = train_test_split_index(len(t), train_frac=train_frac)
    t_tr, q_tr, PI_tr, Pwf_tr = t[:split_idx], q[:split_idx], PI[:split_idx], Pwf[:split_idx]

    # 3) Define models to test
    models = [
        ("Exponential", fit_exponential),
        ("Static",      fit_static),
        ("Arps",        fit_arps),
        ("DynamicEnsemble", fit_dynamic),
        ("CombinedExpArps", fit_combined),
        ("WeightedEnsemble", fit_weighted),
    ]

    # 4) Fit each and plot
    for name, fit_fn in models:
        print(f"\n=== Fitting {name} on training data (first {int(train_frac*100)}%) ===")
        params = fit_fn(t_tr, q_tr, PI_tr, Pwf_tr)
        print("Params:", {k: (f"{v:.6g}" if isinstance(v, float) else v) for k, v in params.items()})

        # base closed-loop forecast
        strat = make_strategy(name, params)
        t_full, q_pred_base = predict_closed_loop_with_strategy(
            strat, t, PI, Pwf, split_idx, n_ahead=n_ahead,
            pi_mode="hold_last", pwf_mode="hold_last", window=30
        )

        # If horizon extended, pad observed for plotting
        t_plot, q_plot = t, q
        if t_full.size > t.size:
            q_obs_full = np.concatenate([q, np.full(t_full.size - t.size, np.nan)])
            t_plot, q_plot = t_full, q_obs_full

        # 5 scenario curves (deterministic)
        scenario_curves = build_scenario_curves(name, params, t_plot, PI, Pwf, split_idx, n_ahead=0)

        fig = build_figure_closed_loop(t_plot, q_plot, q_pred_base, split_idx, scenario_curves=scenario_curves)
        fig.update_layout(title=dict(text=f"<b>Volve F-12: {name} — Closed-Loop Forecast</b>"))
        fig.show()

if __name__ == "__main__":
    _ = main(load_via_pipeline=True)


In [ ]:
# =========================
# NEW: time-aware split helper (train 40%, tune 10%, test 50% by default)
# =========================
def split_indices(n: int, train_frac: float = 0.40, tune_frac: float = 0.10) -> tuple[int, int]:
    """Returns (train_end_idx, tune_end_idx). Test starts at tune_end_idx."""
    train_end = max(1, int(np.floor(n * train_frac)))
    tune_end = max(train_end + 1, int(np.floor(n * (train_frac + tune_frac))))
    tune_end = min(tune_end, n)  # clamp
    return train_end, tune_end


# =========================
# NEW: metrics
# =========================
def mae(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask])))

def rmse(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(np.mean((y_true[mask] - y_pred[mask])**2)))

def SMAPE(y_true, y_pred, eps: float = 1e-8):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt, yp = y_true[mask], y_pred[mask]
    denom = (np.abs(yt) + np.abs(yp)).clip(min=eps)
    return float(np.mean(2.0 * np.abs(yt - yp) / denom))

def mase(y_true, y_pred, y_train_last):
    """MASE using naive=constant(last training value) over the evaluation window."""
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt, yp = y_true[mask], y_pred[mask]
    mae_model = np.mean(np.abs(yt - yp))
    mae_naive = np.mean(np.abs(yt - y_train_last))
    return float(mae_model / max(mae_naive, 1e-8))


# =========================
# UPDATED: figure builder to shade Training & Validation (Tune) regions
# =========================
def build_figure_closed_loop(
    t: np.ndarray,
    q_actual: np.ndarray,
    q_pred_tail: np.ndarray,
    split_idx: int,
    scenario_curves: list[tuple[str, np.ndarray, str]] | None = None,  # (label, q_pred, dash)
    tune_end_idx: int | None = None,
) -> go.Figure:
    panel_titles = ["<b>Observed Production Data</b>", "<b>Closed-Loop Forecast vs. Observed</b>"]
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=panel_titles)

    # Left = observed
    add_observed_data_panel(fig, t, q_actual)

    # Right = observed + base forecast
    fig.add_trace(
        go.Scatter(
            x=t, y=q_actual, mode="lines",
            name="Observed Production",
            line=dict(color=PALETTE["artifact"], width=STYLE["line_width"], shape="spline"),
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(
            x=t, y=q_pred_tail, mode="lines",
            name="Closed-Loop Forecast",
            line=dict(color=PALETTE["process"], width=STYLE["line_width"], shape="spline"),
        ),
        row=1, col=2
    )

    # Optional extra scenario curves
    if scenario_curves:
        for label, q_curve, dash in scenario_curves:
            fig.add_trace(
                go.Scatter(
                    x=t, y=q_curve, mode="lines", name=label,
                    line=dict(color=PALETTE["process"], width=2, dash=dash, shape="spline"),
                ),
                row=1, col=2
            )

    # Regions: Training (gray), Validation/Tune (light yellow), Test (unshaded)
    t_split = float(t[split_idx - 1]) if split_idx > 0 else float(t[0])
    if tune_end_idx is None:
        tune_end_idx = split_idx  # no tune shading if not provided
    t_tune_end = float(t[tune_end_idx - 1]) if tune_end_idx > split_idx else t_split

    for c in (1, 2):
        # Train region
        fig.add_vrect(x0=t.min(), x1=t_split,
                      fillcolor=PALETTE["inputOutput"], opacity=0.12, line_width=0, row=1, col=c)
        # Validation/Tune region (use startEnd color lightly)
        if t_tune_end > t_split:
            fig.add_vrect(x0=t_split, x1=t_tune_end,
                          fillcolor=PALETTE["startEnd"], opacity=0.10, line_width=0, row=1, col=c)
        # Split markers
        fig.add_vline(x=t_split, line=dict(color=PALETTE["startEnd"], width=STYLE["event_line"], dash="dash"), row=1, col=c)
        if t_tune_end > t_split:
            fig.add_vline(x=t_tune_end, line=dict(color=PALETTE["startEnd"], width=STYLE["event_line"], dash="dot"), row=1, col=c)

    # Axes + layout
    y_max = float(np.nanmax([q_actual, q_pred_tail] + ([c[1] for c in scenario_curves] if scenario_curves else [])) * 1.10)
    style_axes(fig, (float(t.min()), float(t.max())), y_max)

    # Region labels
    x_train_mid = t.min() + 0.5 * (t_split - t.min())
    x_tune_mid = t_split + 0.5 * (t_tune_end - t_split) if t_tune_end > t_split else t_split
    x_test_mid = t_tune_end + 0.5 * (float(t.max()) - t_tune_end) if t_tune_end < float(t.max()) else float(t.max())
    for c in (1, 2):
        fig.add_annotation(x=x_train_mid, y=y_max * 0.97, text="<b>Training</b>",
                           showarrow=False, font=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]), row=1, col=c)
        if t_tune_end > t_split:
            fig.add_annotation(x=x_tune_mid, y=y_max * 0.97, text="<b>Validation</b>",
                               showarrow=False, font=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]), row=1, col=c)
        fig.add_annotation(x=x_test_mid, y=y_max * 0.97, text="<b>Test</b>",
                           showarrow=False, font=dict(size=STYLE["tick_font"], color=PALETTE["stroke"]), row=1, col=c)

    fig.update_layout(
        title=dict(
            text="<b>Volve F-12: Exponential Pressure Decay — Closed-Loop Forecast</b>",
            y=0.98, x=0.5, xanchor="center", yanchor="top",
            font=dict(size=STYLE["main_title"], color=PALETTE["stroke"]),
        ),
        width=STYLE["w"], height=STYLE["h"],
        paper_bgcolor=PALETTE["bg"], plot_bgcolor=PALETTE["bg"],
        margin=dict(l=80, r=50, t=110, b=60),
        legend=dict(
            orientation="h", yanchor="bottom", y=-0.30, xanchor="center", x=0.5,
            font=dict(size=STYLE["legend_font"], color=PALETTE["stroke"]),
            bgcolor="rgba(255,255,255,0)",
        ),
        showlegend=True,
    )
    set_panel_title_fonts(fig, panel_titles)
    return fig


In [ ]:
# =========================
# UPDATED main: fit all models, select by Validation (10%), plot w/ tune shading
# =========================
def main(
    df_f12: pd.DataFrame | None = None,
    train_frac: float = 0.40,
    tune_frac: float = 0.10,
    load_via_pipeline: bool = True,
    dataset_name: str = "VOLVE",
    target_well: str = "15/9-F-14",
    n_ahead: int = 0,                   # extend beyond dataset if desired
    selection_metric: str = "SMAPE",    # "MAE" | "RMSE" | "SMAPE" | "MASE"
):
    # 1) Load data (same as before)
    if df_f12 is not None:
        df = df_f12.copy()
        print("✅ Usando DataFrame fornecido (df_f12).")
    else:
        if not load_via_pipeline:
            raise ValueError("Forneça df_f12 ou habilite load_via_pipeline=True.")
        from common.config_wells import get_data_sources
        from data.data_loading import DataSource
        from common.batch_preprocessing import load_and_preprocess_data

        volve_config = next((ds for ds in get_data_sources() if ds.get("name") == dataset_name), None)
        if volve_config is None:
            raise ValueError(f"Config '{dataset_name}' não encontrada.")
        df = load_and_preprocess_data(
            DataSource, config=volve_config,
            selected_features=volve_config.get("features"),
            well=target_well
        )
        print(f"✅ Dados carregados: {len(df):,d} linhas.")

    # 2) Extract + time-aware splits
    t, q, PI, Pwf = extract_series(df)
    n = len(t)
    train_end, tune_end = split_indices(n, train_frac=train_frac, tune_frac=tune_frac)
    print(f"Split -> Train: 0:{train_end}  |  Tune: {train_end}:{tune_end}  |  Test: {tune_end}:{n}")

    t_tr, q_tr, PI_tr, Pwf_tr = t[:train_end], q[:train_end], PI[:train_end], Pwf[:train_end]

    # 3) Define models to test (same list as before)
    models = [
        ("Exponential",       fit_exponential),
        ("Static",            fit_static),
        ("Arps",              fit_arps),
        ("DynamicEnsemble",   fit_dynamic),
        ("CombinedExpArps",   fit_combined),
        ("WeightedEnsemble",  fit_weighted),
    ]

    # 4) Fit on training, forecast closed-loop for full axis, compute tune metrics
    results = []
    for name, fit_fn in models:
        print(f"\n=== Fitting {name} on Training (first {int(train_frac*100)}%) ===")
        params = fit_fn(t_tr, q_tr, PI_tr, Pwf_tr)
        print("Params:", {k: (f"{v:.6g}" if isinstance(v, float) else v) for k, v in params.items()})

        strat = make_strategy(name, params)
        t_full, q_pred_full = predict_closed_loop_with_strategy(
            strat, t, PI, Pwf, train_end, n_ahead=n_ahead,
            pi_mode="hold_last", pwf_mode="hold_last", window=30
        )

        # Align observed to t_full if horizon was extended
        q_obs_plot = q
        t_plot = t
        if t_full.size > t.size:
            q_obs_plot = np.concatenate([q, np.full(t_full.size - t.size, np.nan)])
            t_plot = t_full

        # Evaluation on TUNE slice ONLY
        y_true_tune = q[train_end:tune_end]
        y_pred_tune = q_pred_full[train_end:tune_end]

        # Metrics
        y_train_last = q[train_end - 1] if train_end > 0 else q[0]
        metrics = {
            "MAE": mae(y_true_tune, y_pred_tune),
            "RMSE": rmse(y_true_tune, y_pred_tune),
            "SMAPE": SMAPE(y_true_tune, y_pred_tune),
            "MASE": mase(y_true_tune, y_pred_tune, y_train_last),
        }
        score = metrics.get(selection_metric, metrics["SMAPE"])
        results.append({"name": name, "params": params, "metrics": metrics, "score": score,
                        "t_plot": t_plot, "q_obs_plot": q_obs_plot, "q_pred_full": q_pred_full})

        # Build 5 deterministic scenario curves for this model (no randomness)
        scenario_curves = build_scenario_curves(name, params, t_plot, PI, Pwf, train_end, n_ahead=0)

        # Plot with Training + Validation shading
        fig = build_figure_closed_loop(
            t_plot, q_obs_plot, q_pred_full, train_end,
            scenario_curves=scenario_curves, tune_end_idx=tune_end
        )
        fig.update_layout(title=dict(text=f"<b>Volve F-12: {name} — Closed-Loop (Train/Tune/Test)</b>"))
        fig.show()

    # 5) Rank models by chosen selection metric on the TUNE window
    results_sorted = sorted(results, key=lambda r: r["score"])
    print("\n=== Validation (Tune) Ranking — metric:", selection_metric, "===")
    for i, r in enumerate(results_sorted, 1):
        m = r["metrics"]
        print(f"{i:2d}. {r['name']:18s}  {selection_metric}={r['score']:.4f}  "
              f"(MAE={m['MAE']:.2f}, RMSE={m['RMSE']:.2f}, SMAPE={m['SMAPE']:.4f}, MASE={m['MASE']:.3f})")

    # Optionally return artifacts for further steps
    return {"results": results_sorted, "train_end": train_end, "tune_end": tune_end}

if __name__ == "__main__":
    _ = main(load_via_pipeline=True, selection_metric="SMAPE")


In [ ]:
# =========================
# Fixed-hyperparameter fit wrappers
# =========================
def fit_arps_fixed_b(t_tr, q_tr, PI_tr, Pwf_tr, b_fixed: float) -> dict:
    """Fit Arps with b fixed; optimize P_reservoir and decay_rate on training only."""
    P_res0, decay0 = initial_guess(t_tr, q_tr, PI_tr, Pwf_tr)
    bounds_lo = [1e-3, 1e-8]
    bounds_hi = [1e6, 1.0]

    # Closure: only two free params; b is fixed via closure
    def _wrap(xdata, P_reservoir, decay_rate):
        return q_model_wrapper_arps(xdata, P_reservoir, decay_rate, float(b_fixed))

    popt, _ = curve_fit(
        _wrap,
        xdata=(t_tr, PI_tr, Pwf_tr),
        ydata=q_tr,
        p0=[P_res0, max(decay0, 1e-5)],
        bounds=(bounds_lo, bounds_hi),
        maxfev=30000,
    )
    return {
        "P_reservoir": float(popt[0]),
        "decay_rate": float(popt[1]),
        "b_factor": float(b_fixed),
    }


def fit_combined_fixed_b(t_tr, q_tr, PI_tr, Pwf_tr, b_fixed: float) -> dict:
    """Fit CombinedExpArps with b fixed; optimize P_reservoir and decay_rate."""
    P_res0, decay0 = initial_guess(t_tr, q_tr, PI_tr, Pwf_tr)
    bounds_lo = [1e-3, 1e-8]
    bounds_hi = [1e6, 1.0]

    def _wrap(xdata, P_reservoir, decay_rate):
        return q_model_wrapper_combined(xdata, P_reservoir, decay_rate, float(b_fixed))

    popt, _ = curve_fit(
        _wrap,
        xdata=(t_tr, PI_tr, Pwf_tr),
        ydata=q_tr,
        p0=[P_res0, max(decay0, 1e-5)],
        bounds=(bounds_lo, bounds_hi),
        maxfev=30000,
    )
    return {
        "P_reservoir": float(popt[0]),
        "decay_rate": float(popt[1]),
        "b_factor": float(b_fixed),
    }


def fit_weighted_fixed_alpha(t_tr, q_tr, PI_tr, Pwf_tr, alpha_fixed: float) -> dict:
    """Fit WeightedEnsemble with alpha fixed; optimize P_reservoir and decay_rate."""
    P_res0, decay0 = initial_guess(t_tr, q_tr, PI_tr, Pwf_tr)
    bounds_lo = [1e-3, 1e-8]
    bounds_hi = [1e6, 1.0]

    def _wrap(xdata, P_reservoir, decay_rate):
        return q_model_wrapper_weighted(xdata, P_reservoir, decay_rate, float(alpha_fixed))

    popt, _ = curve_fit(
        _wrap,
        xdata=(t_tr, PI_tr, Pwf_tr),
        ydata=q_tr,
        p0=[P_res0, max(decay0, 1e-5)],
        bounds=(bounds_lo, bounds_hi),
        maxfev=30000,
    )
    return {
        "P_reservoir": float(popt[0]),
        "decay_rate": float(popt[1]),
        "alpha": float(alpha_fixed),
    }

In [ ]:
# =========================
# Grid search over knobs -> rank Top-5 on Validation
# =========================
def evaluate_on_tune(y_true_tune: np.ndarray, y_pred_tune: np.ndarray, y_train_last: float) -> dict:
    return {
        "MAE": mae(y_true_tune, y_pred_tune),
        "RMSE": rmse(y_true_tune, y_pred_tune),
        "SMAPE": SMAPE(y_true_tune, y_pred_tune),
        "MASE": mase(y_true_tune, y_pred_tune, y_train_last),
    }

def grid_search_topk_for_model(
    name: str,
    t: np.ndarray, q: np.ndarray, PI: np.ndarray, Pwf: np.ndarray,
    train_end: int, tune_end: int,
    grids: dict,                 # e.g., {"b_factor":[...]} or {"alpha":[...]}
    topk: int = 5,
    selection_metric: str = "SMAPE",
    n_ahead: int = 0,
) -> dict:
    """
    Returns dict with:
      - 'topk': list of candidates, each {params, score, metrics, q_pred}
      - 'best': first entry in 'topk'
    """
    t_tr, q_tr, PI_tr, Pwf_tr = t[:train_end], q[:train_end], PI[:train_end], Pwf[:train_end]
    candidates = []

    # Build list of hyperparameter settings for this model
    hp_items = list(grids.items()) if grids else []
    if name in {"Arps", "CombinedExpArps", "WeightedEnsemble"} and not hp_items:
        raise ValueError(f"Grid for {name} is empty; provide 'b_factor' or 'alpha' values.")

    # Iterate grid
    if name == "Arps":
        for b in grids.get("b_factor", []):
            params = fit_arps_fixed_b(t_tr, q_tr, PI_tr, Pwf_tr, b_fixed=float(b))
            strat = make_strategy(name, params)
            t_full, q_pred = predict_closed_loop_with_strategy(
                strat, t, PI, Pwf, train_end, n_ahead=n_ahead,
                pi_mode="hold_last", pwf_mode="hold_last", window=30
            )
            y_true_tune = q[train_end:tune_end]
            y_pred_tune = q_pred[train_end:tune_end]
            m = evaluate_on_tune(y_true_tune, y_pred_tune, q[train_end - 1])
            candidates.append({"params": params, "metrics": m, "score": m.get(selection_metric, m["SMAPE"]), "q_pred": q_pred})

    elif name == "CombinedExpArps":
        for b in grids.get("b_factor", []):
            params = fit_combined_fixed_b(t_tr, q_tr, PI_tr, Pwf_tr, b_fixed=float(b))
            strat = make_strategy(name, params)
            t_full, q_pred = predict_closed_loop_with_strategy(
                strat, t, PI, Pwf, train_end, n_ahead=n_ahead,
                pi_mode="hold_last", pwf_mode="hold_last", window=30
            )
            y_true_tune = q[train_end:tune_end]
            y_pred_tune = q_pred[train_end:tune_end]
            m = evaluate_on_tune(y_true_tune, y_pred_tune, q[train_end - 1])
            candidates.append({"params": params, "metrics": m, "score": m.get(selection_metric, m["SMAPE"]), "q_pred": q_pred})

    elif name == "WeightedEnsemble":
        for a in grids.get("alpha", []):
            params = fit_weighted_fixed_alpha(t_tr, q_tr, PI_tr, Pwf_tr, alpha_fixed=float(a))
            strat = make_strategy(name, params)
            t_full, q_pred = predict_closed_loop_with_strategy(
                strat, t, PI, Pwf, train_end, n_ahead=n_ahead,
                pi_mode="hold_last", pwf_mode="hold_last", window=30
            )
            y_true_tune = q[train_end:tune_end]
            y_pred_tune = q_pred[train_end:tune_end]
            m = evaluate_on_tune(y_true_tune, y_pred_tune, q[train_end - 1])
            candidates.append({"params": params, "metrics": m, "score": m.get(selection_metric, m["SMAPE"]), "q_pred": q_pred})

    else:
        # Models without a knob -> single candidate using default fit
        fit_fn = {
            "Exponential":      fit_exponential,
            "Static":           fit_static,
            "DynamicEnsemble":  fit_dynamic,
        }.get(name)
        if fit_fn is None:
            raise ValueError(f"Unknown model '{name}' or missing grid.")
        params = fit_fn(t_tr, q_tr, PI_tr, Pwf_tr)
        strat = make_strategy(name, params)
        t_full, q_pred = predict_closed_loop_with_strategy(
            strat, t, PI, Pwf, train_end, n_ahead=n_ahead,
            pi_mode="hold_last", pwf_mode="hold_last", window=30
        )
        y_true_tune = q[train_end:tune_end]
        y_pred_tune = q_pred[train_end:tune_end]
        m = evaluate_on_tune(y_true_tune, y_pred_tune, q[train_end - 1])
        candidates.append({"params": params, "metrics": m, "score": m.get(selection_metric, m["SMAPE"]), "q_pred": q_pred})

    # Rank & select
    candidates_sorted = sorted(candidates, key=lambda c: c["score"])
    topk_list = candidates_sorted[:max(1, min(5, len(candidates_sorted)))]
    return {"topk": topk_list, "best": topk_list[0]}


In [ ]:
# =========================
# NEW main: hyperparameter grid selection -> Top-5 curves per model
# =========================
def main_grid_select(
    df_f12: pd.DataFrame | None = None,
    train_frac: float = 0.40,
    tune_frac: float = 0.20,
    load_via_pipeline: bool = True,
    dataset_name: str = "VOLVE",
    target_well: str = "15/9-F-14",
    n_ahead: int = 0,
    selection_metric: str = "SMAPE",
):
    # --- Load data (identical to your existing main) ---
    if df_f12 is not None:
        df = df_f12.copy()
        print("✅ Usando DataFrame fornecido (df_f12).")
    else:
        if not load_via_pipeline:
            raise ValueError("Forneça df_f12 ou habilite load_via_pipeline=True.")
        from common.config_wells import get_data_sources
        from data.data_loading import DataSource
        from common.batch_preprocessing import load_and_preprocess_data

        volve_config = next((ds for ds in get_data_sources() if ds.get("name") == dataset_name), None)
        print(volve_config)
        if volve_config is None:
            raise ValueError(f"Config '{dataset_name}' não encontrada.")
        df = load_and_preprocess_data(
            DataSource, config=volve_config,
            selected_features=volve_config.get("features"),
            well=target_well
        )
        df = df[10:]
        print(f"✅ Dados carregados: {len(df):,d} linhas.")

    # --- Extract & split ---
    t, q, PI, Pwf = extract_series(df)
    n = len(t)
    train_end, tune_end = split_indices(n, train_frac=train_frac, tune_frac=tune_frac)
    print(f"Split -> Train: 0:{train_end} | Tune: {train_end}:{tune_end} | Test: {tune_end}:{n}")

    # --- Define models and grids (tweak values as needed) ---
    model_grids = {
        "Exponential":        {},  # no hyperparameter
        "Static":             {},  # no hyperparameter
        "DynamicEnsemble":    {},  # no hyperparameter
        "Arps":               {"b_factor": [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.6]},
        "CombinedExpArps":    {"b_factor": [0.3, 0.5, 0.8, 1.2]},
        "WeightedEnsemble":   {"alpha":    [0.1, 0.3, 0.5, 0.7, 0.9]},
    }

    line_dashes = ["solid", "dash", "dot", "dashdot", "longdash"]  # for Top-1..Top-5

    # --- Run grid search per model ---
    for name, grids in model_grids.items():
        print(f"\n=== Grid select: {name} ===")
        res = grid_search_topk_for_model(
            name, t, q, PI, Pwf, train_end, tune_end,
            grids=grids, topk=5, selection_metric=selection_metric, n_ahead=n_ahead
        )

        # Prepare plot curves: Top-1 as base, others as scenario overlays
        topk = res["topk"]
        base = topk[0]
        q_pred_base = base["q_pred"]

        # If horizon extended, pad observed for plotting (we keep n_ahead=0 by default)
        t_plot, q_plot = t, q

        # Labels with metric
        scenario_curves = []
        for i, cand in enumerate(topk[1:5], start=2):
            label = f"Top-{i} ({selection_metric}={cand['score']:.4f})"
            scenario_curves.append((label, cand["q_pred"], line_dashes[(i-1) % len(line_dashes)]))

        # Plot with training+validation shading (reuse your figure builder)
        fig = build_figure_closed_loop(
            t_plot, q_plot, q_pred_base, train_end,
            scenario_curves=scenario_curves, tune_end_idx=tune_end
        )
        fig.update_layout(title=dict(text=f"<b>Volve F-12: {name} — Grid Top-5 (Tune={selection_metric})</b>"))
        fig.show()

        # Print ranked table for this model
        print(f"Top candidates for {name} (by {selection_metric} on tune):")
        for rank, cand in enumerate(topk, 1):
            pm = ", ".join([f"{k}={v:.4g}" for k, v in cand["params"].items()])
            m = cand["metrics"]
            print(f" {rank:>2}. {pm}  | score={cand['score']:.4f}  "
                  f"(MAE={m['MAE']:.3f}, RMSE={m['RMSE']:.3f}, SMAPE={m['SMAPE']:.4f}, MASE={m['MASE']:.3f})")

if __name__ == "__main__":
    # Keep your old main() intact. Call the new one when you want grid selection:
    _ = main_grid_select(load_via_pipeline=True, selection_metric="SMAPE")
